# TUMSOEV LatentSync 1.5 — ручной бесплатный тест

Чистый ноутбук без встроенных частных фото и голосов. Вы сами загружаете видео и аудио. Версия 1.5 требует примерно 8 ГБ VRAM и подходит для теста на T4 лучше, чем версия 1.6.

Перед запуском: **Runtime → Change runtime type → T4 GPU**, затем **Runtime → Run all**.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
if torch.cuda.is_available(): print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
assert torch.cuda.is_available(), 'Включите T4 GPU: Runtime → Change runtime type → T4 GPU'

## 1. Установка официального LatentSync 1.5
Модель и веса занимают несколько гигабайт. Первая установка может быть долгой.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libgl1 git-lfs
!rm -rf /content/LatentSync
!git clone -q https://github.com/bytedance/LatentSync.git /content/LatentSync
%cd /content/LatentSync
!pip -q install --upgrade pip
!pip -q install -r requirements.txt
!mkdir -p checkpoints/whisper
!huggingface-cli download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints
print('LatentSync 1.5 готов')

## 2. Загрузите одно видео и одно аудио
Для стабильности лицо должно быть достаточно крупным, хорошо освещённым и не закрытым рукой.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, os

os.makedirs('/content/tumsoev_latentsync', exist_ok=True)
uploaded = files.upload()
video_ext = {'.mp4','.mov','.webm','.mkv','.avi'}
audio_ext = {'.wav','.mp3','.m4a','.aac','.flac','.ogg'}
videos = [n for n in uploaded if Path(n).suffix.lower() in video_ext]
audios = [n for n in uploaded if Path(n).suffix.lower() in audio_ext]
assert videos and audios, 'Нужно выбрать одно видео и одно аудио.'
source_video = '/content/' + Path(videos[0]).name
source_audio = '/content/' + Path(audios[0]).name
Path(source_video).write_bytes(uploaded[videos[0]])
Path(source_audio).write_bytes(uploaded[audios[0]])
video = '/content/tumsoev_latentsync/input_5s_25fps.mp4'
audio = '/content/tumsoev_latentsync/voice_5s.wav'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',source_video,'-t','5','-vf','scale=min(720\,iw):-2,fps=25,format=yuv420p','-an','-c:v','libx264','-crf','18',video], check=True)
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',source_audio,'-t','5','-af','apad,atrim=0:5','-ar','16000','-ac','1',audio], check=True)
print('Файлы подготовлены:', video, audio)

## 3. Запустите lip‑sync
Для первого теста: 20 шагов и guidance 1.5. Если лицо дрожит, попробуйте 1.2; если губы совпадают слабо — 1.8.

In [ ]:
%cd /content/LatentSync
OUT = '/content/TUMSOEV_LATENTSYNC_5s.mp4'
!python -m scripts.inference --unet_config_path configs/unet/stage2.yaml --inference_ckpt_path checkpoints/latentsync_unet.pt --inference_steps 20 --guidance_scale 1.5 --video_path /content/tumsoev_latentsync/input_5s_25fps.mp4 --audio_path /content/tumsoev_latentsync/voice_5s.wav --video_out_path /content/TUMSOEV_LATENTSYNC_5s.mp4 --seed 1247 --enable_deepcache
import os
assert os.path.exists(OUT), 'LatentSync не создал MP4 — прочитайте ошибку выше.'
print('ГОТОВО:', OUT)

In [ ]:
from IPython.display import Video, display
from google.colab import files
display(Video(OUT, embed=True))
files.download(OUT)